In [1]:

import numpy as np
from scipy.optimize import (
    linprog,
    milp,
    Bounds,
    LinearConstraint
)


def solve_transport(supply, demand, cost):
    supply = np.asarray(supply, dtype=float)
    demand = np.asarray(demand, dtype=float)
    cost = np.asarray(cost, dtype=float)

    m, n = cost.shape

    assert len(supply) == m
    assert len(demand) == n
    assert np.all(supply >= 0)
    assert np.all(demand >= 0)
    assert np.isclose(supply.sum(), demand.sum())

    # --------------------------------
    # Kantorovich: Linear programming
    # --------------------------------
    A_eq = []
    b_eq = []

    for i in range(m):
        row = np.zeros((m, n))
        row[i, :] = 1
        A_eq.append(row.ravel())
        b_eq.append(supply[i])

    for j in range(n):
        row = np.zeros((m, n))
        row[:, j] = 1
        A_eq.append(row.ravel())
        b_eq.append(demand[j])

    kant = linprog(
        c=cost.ravel(),
        A_eq=np.array(A_eq),
        b_eq=np.array(b_eq),
        bounds=(0, None),
        method="highs"
    )

    print("KANTOROVICH")

    if kant.success:
        plan = kant.x.reshape(m, n)
        print("Optimal plan:")
        print(np.round(plan, 6))
        print("Optimal cost:", round(kant.fun, 6))
    else:
        print("No solution:", kant.message)

    # --------------------------------
    # Monge: Mixed-integer programming
    # --------------------------------
    # z[i,j] = 1 if bakery i sends
    # all its supply to cafe j.

    A = []
    rhs = []

    # Every bakery chooses exactly one cafe
    for i in range(m):
        row = np.zeros((m, n))
        row[i, :] = 1
        A.append(row.ravel())
        rhs.append(1)

    # Each cafe receives exactly its demand
    for j in range(n):
        row = np.zeros((m, n))
        row[:, j] = supply
        A.append(row.ravel())
        rhs.append(demand[j])

    monge = milp(
        c=(cost * supply[:, None]).ravel(),
        integrality=np.ones(m * n),
        bounds=Bounds(0, 1),
        constraints=LinearConstraint(
            np.array(A),
            np.array(rhs),
            np.array(rhs)
        )
    )

    print("\nMONGE")

    if monge.success:
        assignment = np.rint(
            monge.x.reshape(m, n)
        ).astype(int)

        plan = supply[:, None] * assignment

        print("Assignment matrix:")
        print(assignment)
        print("Transport plan:")
        print(plan)
        print("Optimal cost:", round(monge.fun, 6))
    else:
        print("No feasible Monge solution.")
        print(monge.message)


# Example 1: Both formulations feasible
print("========== EXAMPLE 1 ==========")

solve_transport(
    supply=[2, 3, 5],
    demand=[5, 5],
    cost=[
        [2, 6],
        [3, 4],
        [6, 1]
    ]
)


# Example 2: Monge infeasible
print("\n========== EXAMPLE 2 ==========")

solve_transport(
    supply=[5, 5],
    demand=[6, 4],
    cost=[
        [2, 6],
        [3, 4]
    ]
)


========== EXAMPLE 1 ==========
KANTOROVICH
Optimal plan:
[[ 2.  0.]
 [ 3.  0.]
 [-0.  5.]]
Optimal cost: 18.0

MONGE
Assignment matrix:
[[1 0]
 [1 0]
 [0 1]]
Transport plan:
[[2. 0.]
 [3. 0.]
 [0. 5.]]
Optimal cost: 18.0

========== EXAMPLE 2 ==========
KANTOROVICH
Optimal plan:
[[5. 0.]
 [1. 4.]]
Optimal cost: 29.0

MONGE
No feasible Monge solution.
The problem is infeasible. (HiGHS Status 8: model_status is Infeasible; primal_status is At lower/fixed bound)


In [2]:
# ============================================================
# ADDITIONAL VERIFICATION: N < M
# ============================================================

print("\n========== EXAMPLE 3: N < M ==========")

# Original Homework 3.1 example
supply = np.array([
    5.0,
    6.0,
    4.0
])

demand = np.array([
    3.0,
    4.0,
    5.0,
    3.0
])

bakery_locations = np.array([
    [1.0, 1.0],
    [7.0, 1.0],
    [4.0, 7.0]
])

cafe_locations = np.array([
    [2.0, 3.0],
    [6.0, 3.0],
    [3.0, 6.0],
    [7.0, 6.0]
])

cost = np.linalg.norm(
    bakery_locations[:, None, :]
    -
    cafe_locations[None, :, :],
    axis=2
)

solve_transport(
    supply=supply,
    demand=demand,
    cost=cost
)


========== EXAMPLE 3: N < M ==========
KANTOROVICH
Optimal plan:
[[3. 0. 2. 0.]
 [0. 4. 0. 2.]
 [0. 0. 3. 1.]]
Optimal cost: 43.827724

MONGE
No feasible Monge solution.
The problem is infeasible. (HiGHS Status 8: model_status is Infeasible; primal_status is At lower/fixed bound)
